# 第 3 章习题与解答

> 本章习题聚焦参数量计算和超参数理解。建议先自己算,再展开答案。

## Exercise 3.1(易)

**题目**:把 `hidden_size` 改成 512,`num_hidden_layers` 改成 6(其余配置不变),手算新的总参数量。`head_dim` 和 `intermediate_size` 会变成多少?

<details><summary><b>参考答案</b></summary>

先推导派生值:

- `head_dim = 512 // 8 = 64`
- `intermediate_size = ⌈512 × π / 64⌉ × 64 = ⌈25.13⌉ × 64 = 26 × 64 = 1664`

逐项计算:

$$P_{\text{attn}} = 512 \times 8 \times 64 + 2 \times 512 \times 4 \times 64 + 8 \times 64 \times 512 + 2 \times 64 = 786{,}432 + 96 = 786{,}528$$

$$P_{\text{ffn}} = 3 \times 512 \times 1664 = 2{,}555{,}904$$

$$P_{\text{norm}} = 2 \times 512 = 1024$$

$$P_{\text{layer}} = 786{,}528 + 2{,}555{,}904 + 1024 = 3{,}343{,}488$$

$$P_{\text{total}} = 6400 \times 512 + 6 \times 3{,}343{,}488 + 512 = 3{,}276{,}800 + 20{,}060{,}928 + 512 = 23{,}338{,}240$$

**约 23.3M**,比原配置的 63.9M 小了约 63.5%。参数量的下降主要来自两个方面:embedding 层从 4.9M 降到 3.3M,以及每层从 7.37M 降到 3.34M 且层数也少了 2 层。

</details>

In [ ]:
import math

# 验算 Exercise 3.1
d, L, V = 512, 6, 6400
n_q, n_kv = 8, 4
d_h = d // n_q  # 64
d_ff = math.ceil(d * math.pi / 64) * 64  # 1664

P_embed = V * d
P_attn = d*n_q*d_h + 2*d*n_kv*d_h + n_q*d_h*d + 2*d_h
P_ffn = 3 * d * d_ff
P_norm = 2 * d
P_layer = P_attn + P_ffn + P_norm
P_total = P_embed + L * P_layer + d

print(f"head_dim = {d_h}")
print(f"intermediate_size = {d_ff}")
print(f"P_embed = {P_embed:,}")
print(f"P_layer = {P_layer:,}")
print(f"  P_attn = {P_attn:,}")
print(f"  P_ffn  = {P_ffn:,}")
print(f"  P_norm = {P_norm:,}")
print(f"P_total = {P_total:,} = {P_total/1e6:.2f}M")

## Exercise 3.2(中)

**题目**:解释为什么 GQA 的 `num_key_value_heads=4` 比 `=8`(MHA)省了 KV cache,但**不影响** `q_proj` 的参数量。

<details><summary><b>参考答案</b></summary>

**q_proj 不受影响的原因**:

`q_proj` 的输出维度由 `num_attention_heads × head_dim` 决定。无论 `num_key_value_heads` 是 4 还是 8,`num_attention_heads` 始终是 8:

$$\text{q\_proj}: d \times (n_q \times d_h) = 768 \times (8 \times 96) = 768 \times 768$$

这个矩阵的参数量是 $768^2 = 589{,}824$,与 `num_key_value_heads` 无关。

**KV cache 受影响的原因**:

`k_proj` 和 `v_proj` 的输出维度由 `num_key_value_heads × head_dim` 决定:

| 配置 | k_proj/v_proj 输出 | 每层 KV 参数 | 每 token KV cache |
|---|---|---|---|
| MHA (kv_heads=8) | 8×96=768 | 2×768×768=1,179,648 | 2×8×96=1536 |
| GQA (kv_heads=4) | 4×96=384 | 2×768×384=589,824 | 2×4×96=768 |

GQA 把 KV 头数减半 → k_proj 和 v_proj 的参数减半 → KV cache 元素数减半。

**核心区别**:Q 头数决定了「查询能力」(每个 token 能关注多少个不同的子空间),而 KV 头数只影响「缓存大小」。GQA 的洞察是:多个 Q 头可以共享同一组 KV 而不损失太多质量。

</details>

In [ ]:
# 验证 Exercise 3.2
d = 768
n_q = 8
d_h = 96

for label, n_kv in [("MHA (kv=8)", 8), ("GQA (kv=4)", 4)]:
    q_params = d * (n_q * d_h)      # q_proj: 始终相同
    kv_params = 2 * d * (n_kv * d_h) # k_proj + v_proj
    cache_per_token = 2 * n_kv * d_h # KV cache elements per token per layer
    print(f"{label}:")
    print(f"  q_proj params:       {q_params:>10,}  (不变)")
    print(f"  k+v_proj params:     {kv_params:>10,}")
    print(f"  KV cache/token/layer: {cache_per_token:>8} elements")
    print()

## Exercise 3.3(难)

**题目**:如果要把 minimind 从 dense 变成 MoE(`use_moe=True, num_experts=4, num_experts_per_tok=1`),总参数量和激活参数量分别变成多少?写出计算过程。

<details><summary><b>参考答案</b></summary>

**变化的部分**:只有每层的 FFN 从 `FeedForward` 变成 `MOEFeedForward`。Attention、Norms、Embedding 都不变。

**Dense FFN 每层参数**:

$$P_{\text{ffn}}^{\text{dense}} = 3 \times d \times d_{\text{ff}} = 3 \times 768 \times 2432 = 5{,}603{,}328$$

**MoE FFN 每层参数**:

$$P_{\text{ffn}}^{\text{moe}} = \underbrace{N_{\text{exp}} \times 3 \times d \times d_{\text{ff}}}_{\text{专家}} + \underbrace{d \times N_{\text{exp}}}_{\text{router}}$$

$$= 4 \times 5{,}603{,}328 + 768 \times 4 = 22{,}413{,}312 + 3{,}072 = 22{,}416{,}384$$

**每层总参数**(Attention + Norms + MoE FFN):

$$P_{\text{layer}}^{\text{moe}} = 1{,}769{,}664 + 1{,}536 + 22{,}416{,}384 = 24{,}187{,}584$$

**总参数**:

$$P_{\text{total}}^{\text{moe}} = 4{,}915{,}200 + 8 \times 24{,}187{,}584 + 768 = 198{,}416{,}640 \approx 198\text{M}$$

**激活参数量**(top-1 路由,每 token 只经过 1 个专家):

每 token 的 FFN 激活参数 = 1 个专家的 FFN = $5{,}603{,}328$(与 dense 相同)

$$P_{\text{active}}^{\text{moe}} = 4{,}915{,}200 + 8 \times (1{,}769{,}664 + 1{,}536 + 5{,}603{,}328) + 768 = 63{,}912{,}192 \approx 64\text{M}$$

**结论**:MoE 模型的总参数是 dense 的 ~3.1 倍(198M vs 64M),但每个 token 的激活参数量与 dense 完全相同(64M)。这就是 MoE「以 3 倍参数容量换取相同计算成本」的核心价值。

</details>

In [ ]:
import math

d, L, V = 768, 8, 6400
n_q, n_kv, d_h = 8, 4, 96
d_ff = math.ceil(d * math.pi / 64) * 64
N_exp = 4

# 不变部分
P_embed = V * d
P_attn = d*n_q*d_h + 2*d*n_kv*d_h + n_q*d_h*d + 2*d_h
P_norm = 2 * d
P_final = d

# Dense FFN
P_ffn_dense = 3 * d * d_ff

# MoE FFN
P_router = d * N_exp
P_ffn_moe = N_exp * P_ffn_dense + P_router

# 层参数
P_layer_dense = P_attn + P_norm + P_ffn_dense
P_layer_moe   = P_attn + P_norm + P_ffn_moe

# 总参数
P_dense = P_embed + L * P_layer_dense + P_final
P_moe   = P_embed + L * P_layer_moe   + P_final

# 激活参数 (top-1): 每层只激活 1 个专家的 FFN
P_layer_active = P_attn + P_norm + P_ffn_dense  # 1 expert = same as dense FFN
P_moe_active = P_embed + L * P_layer_active + P_final

print(f"{'':>20s} {'Dense':>14s} {'MoE':>14s}")
print(f"{'P_embed':>20s} {P_embed:>14,} {P_embed:>14,}")
print(f"{'P_attn/层':>20s} {P_attn:>14,} {P_attn:>14,}")
print(f"{'P_ffn/层':>20s} {P_ffn_dense:>14,} {P_ffn_moe:>14,}")
print(f"{'P_layer':>20s} {P_layer_dense:>14,} {P_layer_moe:>14,}")
print(f"{'P_total':>20s} {P_dense:>14,} {P_moe:>14,}")
print(f"{'  (M)':>20s} {P_dense/1e6:>13.2f}M {P_moe/1e6:>13.2f}M")
print(f"{'P_active (M)':>20s} {P_dense/1e6:>13.2f}M {P_moe_active/1e6:>13.2f}M")
print()
print(f"MoE/Dense 总参数 = {P_moe/P_dense:.2f}x")
print(f"MoE/Dense 激活参数 = {P_moe_active/P_dense:.2f}x (相同!)")